In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import pandas as pd
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# CONFIG
MODEL_PATH = "../models/hand_landmarker.task"
CSV_PATH = "../data/bukva/annotations.csv"
VIDEO_DIR = "../data/bukva/trimmed/"
OUTPUT_DIR = "../data/bukva_processed/processed_landmarks_3d/"

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH, sep="\t")
df["file_path"] = VIDEO_DIR + df["attachment_id"] + ".mp4"

# MODEL INIT
def initialize_detector():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=2,
        running_mode=vision.RunningMode.VIDEO
    )
    return vision.HandLandmarker.create_from_options(options)

# NORMALIZATION
def normalize_hand(hand_landmarks):
    coords = np.array([[lm.x, lm.y] for lm in hand_landmarks])

    # wrist -> origin
    coords -= coords[0]

    # scale normalize
    max_val = np.max(np.abs(coords))
    if max_val > 0:
        coords /= max_val

    return coords

# HAND ASSIGNMENT
def assign_hands(result, prev_left, prev_right):
    current_left, current_right = None, None

    if not result.hand_landmarks:
        return None, None

    hands = result.hand_landmarks

    if prev_left is None and prev_right is None:
        for h in hands:
            if h[0].x < 0.5:
                current_left = h
            else:
                current_right = h
    else:
        for h in hands:
            wrist = h[0]

            d_l = (wrist.x - prev_left[0])**2 + (wrist.y - prev_left[1])**2 if prev_left else 1e9
            d_r = (wrist.x - prev_right[0])**2 + (wrist.y - prev_right[1])**2 if prev_right else 1e9

            if d_l < d_r:
                current_left = h
            else:
                current_right = h

    return current_left, current_right

# FRAME VECTOR BUILDER
def build_frame_vector(hand_obj):
    if hand_obj is None:
        return [0] + [0] * 42, None

    norm = normalize_hand(hand_obj)
    wrist_pos = (hand_obj[0].x, hand_obj[0].y)

    return [1] + norm.flatten().tolist(), wrist_pos

# PROCESS SINGLE VIDEO
def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Could not open: {video_path}")
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        print(f"Empty video: {video_path}")
        return None

    frame_indices = np.linspace(0, total_frames - 1, 24, dtype=int)

    hand_detector = initialize_detector()

    last_processed_timestamp = -1
    all_landmarks = []
    prev_l, prev_r = None, None
    last_valid = None

    for t, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = cap.read()

        if not success:
            if last_valid is not None:
                all_landmarks.append(last_valid.copy())
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        fps = cap.get(cv2.CAP_PROP_FPS)
        timestamp_ms = int(frame_idx * 1000 / fps)
        if timestamp_ms <= last_processed_timestamp:
            timestamp_ms = last_processed_timestamp + 1

        last_processed_timestamp = timestamp_ms

        result = hand_detector.detect_for_video(
            mp_image,
            timestamp_ms=timestamp_ms
        )

        c_l, c_r = assign_hands(result, prev_l, prev_r)

        l_vec, prev_l = build_frame_vector(c_l)
        r_vec, prev_r = build_frame_vector(c_r)

        frame_vec = l_vec + r_vec
        all_landmarks.append(frame_vec)
        last_valid = frame_vec

    cap.release()
    hand_detector.close()

    # pad to 24 frames
    while len(all_landmarks) < 24:
        if last_valid is not None:
            all_landmarks.append(last_valid.copy())
        else:
            all_landmarks.append([0] * 86)

    return np.array(all_landmarks)


# MAIN LOOP (CSV -> NPY)
for i, row in df.iterrows():
    video_path = row["file_path"]
    attachment_id = row["attachment_id"]

    print(f"[{i+1}/{len(df)}] Processing {attachment_id}")

    data = process_video(video_path)

    if data is None:
        continue

    save_path = os.path.join(OUTPUT_DIR, f"{attachment_id}.npy")
    np.save(save_path, data)

print("DONE ✔")